# Aquaplanet with customized initial conditions

This notebook demonstrates how to change configuration of an aquaplanet simulation.

The code will be largely the same until the section "Customize Initial Condition" and "Run Couple Model".

In [ ]:
from pathlib import Path

import jcm
from jcm.physics.speedy.speedy_coords import get_speedy_coords
from jcm.terrain import TerrainData
from jcm.forcing import ForcingData
from importlib import resources
import jax_datetime as jdt

from jem.components import JCM, SlabOceanModel
from jem.mapping import BasicMapper
from jem.base.coupler import Coupler
import jem.utils.tree_tools as tree_tools

use_ipython = 'get_ipython' in globals()

## Configurations

In [ ]:
start_datetime = jdt.to_datetime("2000-01-01")
coupling_timestep = jdt.to_timedelta(1, "day")
simulation_name = "02_aquaplanet_customized_initial_condition"
output_dir = (Path("output") / simulation_name).resolve()
output_dir.mkdir(exist_ok=True, parents=True)
output_figures = {
    "initial_condition": output_dir / "initial_sst.png",
    "animation": output_dir / "animation_humidity_sst.gif",
}
one_second = jdt.to_timedelta(1, "second")

## Creating Flux and Scalar Exchange between Components

In [ ]:
mapper = BasicMapper()
mapper.add_mapping(
    source = ("atm", "derived.total_heat_flux"),
    target = ("ocn", "forcing.total_heat_flux"),
    regridder = lambda x: x,  # identity is default
)
mapper.add_mapping(
    source = ("ocn", "state.sea_surface_temperature"),
    target = ("atm", "forcing.sea_surface_temperature"),
)

## Create Components

In [ ]:
atm_model = jcm.model.Model(
    coords=get_speedy_coords(),  # T31 spectral resolution with 8 vertical levels
    start_date=start_datetime,
)

atm_model = JCM.make_jem_compatible(
    atm_model,
    coupling_timestep=coupling_timestep,
)

model = Coupler(
    components=dict(
        atm=atm_model,
        ocn=SlabOceanModel(
            start_datetime=start_datetime,
            timestep=coupling_timestep / one_second,
        ),
    ),
    mappers=dict(mapper=mapper),
)

print("Model info: ") 
tree_tools.print_tree(model.get_info(), root="Model")

## Customize Initial Condition

In [ ]:
import jax.numpy as jnp
customized_initial_coupled_carry = model.initialize()
ocean_carry = customized_initial_coupled_carry["ocn"]
ocean_model = model.components["ocn"].raw_component
ocean_carry["state"].sea_surface_temperature += 5 * jnp.sin(ocean_model.longitude_radian * 2) * jnp.cos(ocean_model.latitude_radian)**3

In [ ]:
import matplotlib as mplt
if not use_ipython:
    mplt.use("Agg")
import matplotlib.pyplot as plt


lat = ocean_model.latitude_radian[0, :] * 180.0 / jnp.pi
lon = ocean_model.longitude_radian[:, 0] * 180.0 / jnp.pi
fig, ax = plt.subplots(1, 1)
mappable = ax.contourf(lon, lat, customized_initial_coupled_carry["ocn"]["state"].sea_surface_temperature.transpose() - 273.15, cmap="gnuplot")
cbar = plt.colorbar(mappable, ax=ax)
cbar.set_label("Customized initialial sea surface temperature [${}^\\circ \\mathrm{C}$]")
ax.set_xlabel("Longitude [deg]")
ax.set_ylabel("Latitude [deg]")

print(f"Saving initial condition figure: {output_figures['initial_condition']}")
plt.savefig(output_figures['initial_condition'], dpi=200)

if use_ipython:
    plt.show()



## Run Coupled Model

Use keyword `initial_carry` to set customized initial condition.

In [ ]:
simulation_interval = jdt.to_timedelta(5, "day")
initial_state, final_state, predictions = model.run(
    initial_carry = customized_initial_coupled_carry,
    workflow=["mapper", "atm", "ocn"],
    iterations = int(simulation_interval / coupling_timestep),
)

## Output into NetCDF

In [ ]:
output_dict = model.predictions_to_xarray(predictions)
output_dict_subsample = {}
subsample_skip = 5
for component_name, ds in output_dict.items():
    output_file = output_dir / f"{component_name:s}.nc"
    print(f"Output file: {str(output_file)}, with subsample_skip = {subsample_skip:d}")
    ds = ds.isel(time=slice(None, None, subsample_skip))
    ds.to_netcdf(output_file, engine="netcdf4")
    output_dict_subsample[component_name] = ds

## Visualization
from matplotlib.animation import FuncAnimation
import cartopy.crs as ccrs
from cartopy.util import add_cyclic_point
import numpy as np

output_dict_animation = {
    component_name: _ds.isel(time=slice(None, None, 1))
    for component_name, _ds in output_dict.items()
}

fig = plt.figure(figsize=(10, 6))
ax = plt.axes(projection=ccrs.PlateCarree())

ax.gridlines(draw_labels=True)
cb = None
cf = None
cs = None

def update(frame):
    print(f"Plotting frame={frame:d}")
    global cf, cb, cs 
    _data_q = output_dict_animation["atm"]["specific_humidity"].isel(time=frame, level=0)
    _data_sst = output_dict_animation["ocn"]["sea_surface_temperature"].isel(time=frame) - 273.15
    coords = _data_q.coords
    time_str = _data_q['time'].dt.strftime('%Y-%m-%d').to_numpy().item()
    lat = coords["lat"]
    lon = coords["lon"]

    # Remove contour and contourf if not empty
    cf and cf.remove()
    cs and cs.remove()
    
    # Plot the humidity field for the current time step
    cyclic_data_q, cyclic_lon = add_cyclic_point(_data_q.to_numpy().transpose(), coord=lon)
    mappable = ax.contourf(
        cyclic_lon, lat,
        cyclic_data_q,
        levels=1 + np.linspace(0, 1, 21) * 10,
        transform=ccrs.PlateCarree(), 
        cmap='GnBu',
        extend="both",
    )
    
    cyclic_data_sst, cyclic_lon = add_cyclic_point(_data_sst.to_numpy().transpose(), coord=lon)
    cs = ax.contour(
        cyclic_lon, lat,
        cyclic_data_sst,
        levels=np.arange(-2, 31, 4),
        transform=ccrs.PlateCarree(),
        colors="black",
    )
    ax.clabel(cs, fontsize=12)
    ax.set_title(f"[{time_str:s}]\nSurface specific humidity (shading) and sea surface temperature (contours, ${{}}^\\circ \\mathrm{{C}}$)")
    if cb is None:
        cb = plt.colorbar(ax=ax, mappable=mappable, orientation='vertical', shrink=0.7, pad=0.07)
        cb.set_label("[g/kg]", fontsize=12)
    
    return [cf,]
    
Generate and save
ani = FuncAnimation(fig, update, frames=len(output_dict_animation["atm"].coords["time"]), interval=120, blit=False)
print("Saving animation: ", output_figures['animation'])
ani.save(output_figures['animation'], writer='pillow', dpi=200)

if use_ipython:
    from IPython.display import Image
    display(Image(output_figures['animation']))